## Actualizar una planilla basada en un campo comun
crea un notebook python para cargar las planilals en dos dataframes. 
- un 'df' para 'plan_de_compras_2025.xlsx y 
- un dataframe 'codigos' para la planilla codigos_unicos.xlsx. 

Luego realizar un join entre el dataframe df y el dataframe codigos basado en la columna 'Código presupuestario' de df y la columna 'Codigo' de codigos, para obtener los nombres de los proyectos correspondientes a cada código presupuestario. Reemplazar los valores de la columna 'Nombre Proyecto' en el dataframe df con los valores correspondientes de 'Nombre' en el dataframe codigos basados en el valor de la columna 'Código presupuestario' y 'Codigo' del dataframe codigos.

In [8]:
import pandas as pd

# Cargar las planillas Excel
df = pd.read_excel('plan_de_compras_2025.xlsx')
codigos = pd.read_excel('codigos_unicos.xlsx')

# Asegurar que las columnas de cruce tengan formato string sin espacios extra
df['Código presupuestario_str'] = df['Código presupuestario'].astype(str).str.strip()
codigos['Codigo_str'] = codigos['Codigo'].astype(str).str.strip()

print(df[['Código presupuestario', 'Nombre Proyecto']].head(10))

  Código presupuestario                                    Nombre Proyecto
0                    33  Servicio arriendo de una plataforma de gestión...
1               2206001  Servicio de Mantención y Reparaciones Menores ...
2               2204001           Servicios de Impresión-Seremi Valparaiso
3               2209006  Servicio de Arriendo de Impresoras Multifuncio...
4               2205005       Servicio de Telefonía Fija-Seremi Valparaiso
5                  2207  SERVICIO DE EMPASTES Y ENCUADERNACION- Sección...
6                  2209  Arriendo de impresoras multifuncionales- Secci...
7                  2205  Servicio de Telefonia Satelital- Sección Contr...
8                  2205  Servicio de Telefonia Satelital- Sección Contr...
9                  2208  Servicio del limpieza interna y externa y exte...


Opción A: Mediante merge() (Join explícito)

Realizamos un left join entre df y codigos uniendo la columna 'Código presupuestario_str' con 'Codigo_str'. Luego actualizamos la columna 'Nombre Proyecto' con los nuevos valores traídos desde codigos['Nombre'].

In [5]:
# Realizar el Join entre df y codigos
df_merged = df.merge(
    codigos[['Codigo_str', 'Nombre']], 
    left_on='Código presupuestario_str', 
    right_on='Codigo_str', 
    how='left'
)

# Reemplazar los valores en 'Nombre Proyecto' con el nuevo 'Nombre' obtenido del join
# Se mantiene el valor original en caso de que no haya coincidencia
df['Nombre Proyecto'] = df_merged['Nombre'].fillna(df['Nombre Proyecto'])

# Limpiar columna auxiliar
df.drop(columns=['Código presupuestario_str'], inplace=True)
print(df[['Código presupuestario', 'Nombre Proyecto']].head(10))

  Código presupuestario                                    Nombre Proyecto
0                    33  Servicio arriendo de una plataforma de gestión...
1               2206001               REPARACIONES DEPENDENCIAS MINVU (VEP
2               2204001                             MATERIAL DE ESCRITORIO
3               2209006  Arriendo PC Ampliación (465) - División Inform...
4               2205005                         Servicio de Telefonía Fija
5                  2207              SERVICIO DE EMPASTES Y ENCUADERNACION
6                  2209  Arriendo de impresoras multifuncionales- Secci...
7                  2205                    Servicio de Telefonia Satelital
8                  2205                    Servicio de Telefonia Satelital
9                  2208  Servicio del limpieza interna y externa y exte...


Opción B: Mediante mapeo de diccionario (Forma directa y eficiente)
También puedes mapear directamente los nombres usando un diccionario creado a partir de la planilla codigos:

In [9]:
# Crear diccionario de mapeo {Codigo: Nombre}
mapa_codigos = dict(zip(codigos['Codigo_str'], codigos['Nombre']))

# Reemplazar directamente la columna 'Nombre Proyecto'
df['Nombre Proyecto'] = df['Código presupuestario_str'].map(mapa_codigos).fillna(df['Nombre Proyecto'])

# Limpiar columna auxiliar
df.drop(columns=['Código presupuestario_str'], inplace=True)
print(df[['Código presupuestario', 'Nombre Proyecto']].head(10))

  Código presupuestario                                    Nombre Proyecto
0                    33  Servicio arriendo de una plataforma de gestión...
1               2206001               REPARACIONES DEPENDENCIAS MINVU (VEP
2               2204001                             MATERIAL DE ESCRITORIO
3               2209006  Arriendo PC Ampliación (465) - División Inform...
4               2205005                         Servicio de Telefonía Fija
5                  2207              SERVICIO DE EMPASTES Y ENCUADERNACION
6                  2209  Arriendo de impresoras multifuncionales- Secci...
7                  2205                    Servicio de Telefonia Satelital
8                  2205                    Servicio de Telefonia Satelital
9                  2208  Servicio del limpieza interna y externa y exte...


In [ ]:
# Verificar las primeras filas
print(df[['Código presupuestario', 'Nombre Proyecto']].head(10))

# (Opcional) Guardar el resultado en un nuevo archivo Excel
df.to_excel('plan_de_compras_2025_actualizado.xlsx', index=False)